# DAS7002 – Big Data Technologies
## NYC Yellow Taxi Trip Data Analysis Using Spark Distributed Data Structures and MLlib
**Cardiff Metropolitan University – PRAC1**

---
> **Before you start:** Upload your three Parquet files to this Colab session using the 📁 Files panel on the left, or mount your Google Drive (instructions in Cell 2).

## ⚙️ Setup – Install PySpark

In [ ]:
# Install PySpark in Colab
!pip install pyspark --quiet
!pip install findspark --quiet
print('✅ PySpark installed')

## 📂 Option A – Upload files manually
Use the Files panel (left sidebar 📁) to upload:
- `yellow_tripdata_2025-07.parquet`
- `yellow_tripdata_2025-08.parquet`
- `yellow_tripdata_2025-09.parquet`

Then skip to **Option B** below, or use Option B if your files are in Google Drive.

In [ ]:
# Option B – Mount Google Drive (skip if you uploaded manually)
# from google.colab import drive
# drive.mount('/content/drive')
# PARQUET_PATH = '/content/drive/MyDrive/NYC_Taxi/*.parquet'

# Option A – Files uploaded directly to Colab session
PARQUET_PATH = '/content/yellow_tripdata_2025-*.parquet'

print(f'Parquet path set to: {PARQUET_PATH}')

---
# TASK 1 – Data Preparation (15%)

In [ ]:
import findspark
findspark.init()

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import col, when
from pyspark import StorageLevel

spark = SparkSession.builder \
    .appName('NYC_Taxi_DAS7002') \
    .config('spark.sql.shuffle.partitions', '100') \
    .config('spark.driver.memory', '4g') \
    .config('spark.executor.memory', '4g') \
    .config('spark.sql.adaptive.enabled', 'true') \
    .getOrCreate()

spark.sparkContext.setLogLevel('WARN')
print(f'✅ Spark version: {spark.version}')

In [ ]:
# ── 1.1 Load all three months into a single DataFrame ──────────────────────
df = spark.read.parquet(PARQUET_PATH)

print(f'Total raw records: {df.count():,}')
print(f'Columns: {len(df.columns)}')
df.printSchema()

In [ ]:
# ── 1.2 Drop unnecessary columns ───────────────────────────────────────────
columns_to_drop = [
    'store_and_fwd_flag',
    'congestion_surcharge',
    'airport_fee',
    'improvement_surcharge'
]
# Only drop columns that actually exist in this dataset
columns_to_drop = [c for c in columns_to_drop if c in df.columns]
df = df.drop(*columns_to_drop)

print(f'Columns after dropping: {df.columns}')

In [ ]:
# ── 1.3 Handle missing values ───────────────────────────────────────────────
# Drop rows where critical columns are null
critical_cols = ['tpep_pickup_datetime', 'tpep_dropoff_datetime',
                 'trip_distance', 'total_amount']
df = df.dropna(subset=critical_cols)

# Impute passenger_count with median
median_passengers = df.approxQuantile('passenger_count', [0.5], 0.01)[0]
df = df.fillna({'passenger_count': int(median_passengers)})

print(f'Records after null removal: {df.count():,}')

In [ ]:
# ── 1.4 Remove logically invalid rows ──────────────────────────────────────
df = df.filter(
    (col('trip_distance') > 0) &
    (col('total_amount') > 0) &
    (col('fare_amount') >= 2.50) &
    (col('tpep_dropoff_datetime') > col('tpep_pickup_datetime'))
)

print(f'Records after logic filters: {df.count():,}')

In [ ]:
# ── 1.5 IQR outlier removal on trip_distance and total_amount ──────────────
q1_dist, q3_dist = df.approxQuantile('trip_distance', [0.25, 0.75], 0.01)
iqr_dist = q3_dist - q1_dist

q1_fare, q3_fare = df.approxQuantile('total_amount', [0.25, 0.75], 0.01)
iqr_fare = q3_fare - q1_fare

df = df.filter(
    (col('trip_distance').between(
        q1_dist - 1.5 * iqr_dist, q3_dist + 1.5 * iqr_dist)) &
    (col('total_amount').between(
        q1_fare - 1.5 * iqr_fare, q3_fare + 1.5 * iqr_fare))
)

print(f'Records after outlier removal: {df.count():,}')

In [ ]:
# ── 1.6 Derive new features ─────────────────────────────────────────────────
df = df.withColumn('trip_duration_min',
    (col('tpep_dropoff_datetime').cast('long') -
     col('tpep_pickup_datetime').cast('long')) / 60
)
df = df.withColumn('pickup_hour',  F.hour('tpep_pickup_datetime'))
df = df.withColumn('pickup_day',   F.dayofweek('tpep_pickup_datetime'))
df = df.withColumn('pickup_month', F.month('tpep_pickup_datetime'))

# Remove implausible durations
df = df.filter(col('trip_duration_min').between(1, 180))

# Cache the cleaned DataFrame – avoids re-reading Parquet for every EDA query
df.persist(StorageLevel.MEMORY_AND_DISK)
clean_count = df.count()   # triggers caching
print(f'✅ Final clean records cached: {clean_count:,}')
df.show(5)

---
# TASK 2 – EDA and Performance Optimisation (35%)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', palette='muted')
print('✅ Plotting libraries ready')

In [ ]:
# ── Q1: Trip volume by hour of day and day of week ─────────────────────────
# Wide transformation: groupBy on two columns triggers a shuffle
volume_by_time = df.groupBy('pickup_hour', 'pickup_day') \
    .agg(F.count('*').alias('trip_count')) \
    .orderBy('pickup_day', 'pickup_hour')

pdf_vol = volume_by_time.toPandas()

# Pivot for heatmap
pivot = pdf_vol.pivot(index='pickup_hour', columns='pickup_day', values='trip_count')
pivot.columns = ['Sun','Mon','Tue','Wed','Thu','Fri','Sat']

fig, ax = plt.subplots(figsize=(12, 7))
sns.heatmap(pivot, cmap='YlOrRd', annot=True, fmt='.0f',
            linewidths=0.4, ax=ax, cbar_kws={'label': 'Trip Count'})
ax.set_title('Q1: Trip Volume by Hour of Day and Day of Week', fontsize=14, fontweight='bold')
ax.set_xlabel('Day of Week')
ax.set_ylabel('Hour of Day')
plt.tight_layout()
plt.savefig('q1_volume_heatmap.png', dpi=150)
plt.show()
print('Insight: Clear bimodal weekday demand (08-09h and 17-19h); weekend peaks shift to late evening.')

In [ ]:
# ── Q2: Tip amounts by payment type and trip distance band ─────────────────
from pyspark.ml.feature import Bucketizer

splits = [-float('inf'), 1.0, 3.0, 7.0, 15.0, float('inf')]
bucketizer = Bucketizer(splits=splits,
                        inputCol='trip_distance',
                        outputCol='distance_band')
df_banded = bucketizer.transform(df)

tip_analysis = df_banded.groupBy('payment_type', 'distance_band') \
    .agg(
        F.avg('tip_amount').alias('avg_tip'),
        F.count('*').alias('n_trips')
    ) \
    .orderBy('payment_type', 'distance_band')

pdf_tip = tip_analysis.toPandas()
pdf_tip['distance_label'] = pdf_tip['distance_band'].map(
    {0.0:'<1mi', 1.0:'1-3mi', 2.0:'3-7mi', 3.0:'7-15mi', 4.0:'>15mi'})
pdf_tip['payment_label'] = pdf_tip['payment_type'].map(
    {1:'Card', 2:'Cash', 3:'No Charge', 4:'Dispute'})

fig, ax = plt.subplots(figsize=(10, 5))
for ptype, grp in pdf_tip[pdf_tip['payment_type'].isin([1,2])].groupby('payment_label'):
    ax.plot(grp['distance_label'], grp['avg_tip'], marker='o', label=ptype)
ax.set_title('Q2: Average Tip by Payment Type and Distance Band', fontsize=13, fontweight='bold')
ax.set_xlabel('Distance Band')
ax.set_ylabel('Average Tip ($)')
ax.legend()
plt.tight_layout()
plt.savefig('q2_tip_analysis.png', dpi=150)
plt.show()
print('Insight: Card payers tip proportionally more at all distance bands; cash tips are near zero.')

In [ ]:
# ── Q3: Top 20 pickup-dropoff location pairs ────────────────────────────────
# High-cardinality groupBy – triggers a significant shuffle
od_pairs = df.groupBy('PULocationID', 'DOLocationID') \
    .agg(F.count('*').alias('pair_count')) \
    .orderBy(F.desc('pair_count'))

top20 = od_pairs.limit(20).toPandas()
top20['OD_Pair'] = top20['PULocationID'].astype(str) + ' → ' + top20['DOLocationID'].astype(str)

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(top20['OD_Pair'][::-1], top20['pair_count'][::-1], color='steelblue')
ax.set_title('Q3: Top 20 Most Common Pickup–Dropoff Pairs', fontsize=13, fontweight='bold')
ax.set_xlabel('Number of Trips')
plt.tight_layout()
plt.savefig('q3_od_pairs.png', dpi=150)
plt.show()
top20.head(10)

In [ ]:
# ── Q4: Average speed by hour of day ────────────────────────────────────────
df_speed = df.withColumn('avg_speed_mph',
    col('trip_distance') / (col('trip_duration_min') / 60)
).filter(col('avg_speed_mph').between(1, 80))

speed_by_hour = df_speed.groupBy('pickup_hour') \
    .agg(F.avg('avg_speed_mph').alias('mean_speed')) \
    .orderBy('pickup_hour')

pdf_speed = speed_by_hour.toPandas()

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(pdf_speed['pickup_hour'], pdf_speed['mean_speed'], marker='o', color='coral')
ax.fill_between(pdf_speed['pickup_hour'], pdf_speed['mean_speed'], alpha=0.2, color='coral')
ax.set_title('Q4: Average Speed by Hour of Day', fontsize=13, fontweight='bold')
ax.set_xlabel('Hour of Day')
ax.set_ylabel('Avg Speed (mph)')
ax.set_xticks(range(0, 24))
plt.tight_layout()
plt.savefig('q4_speed_by_hour.png', dpi=150)
plt.show()
print('Insight: Speed peaks in early morning (02-05h), dips sharply in evening rush (17-19h).')

In [ ]:
# ── Q5: Monthly trip volume and average fare trend ──────────────────────────
monthly_trend = df.groupBy('pickup_month') \
    .agg(
        F.count('*').alias('total_trips'),
        F.avg('fare_amount').alias('avg_fare'),
        F.sum('total_amount').alias('total_revenue')
    ).orderBy('pickup_month')

pdf_month = monthly_trend.toPandas()
pdf_month['month_label'] = pdf_month['pickup_month'].map(
    {7:'July', 8:'August', 9:'September'})

fig, ax1 = plt.subplots(figsize=(8, 4))
ax2 = ax1.twinx()
ax1.bar(pdf_month['month_label'], pdf_month['total_trips'],
        color='steelblue', alpha=0.7, label='Trip Count')
ax2.plot(pdf_month['month_label'], pdf_month['avg_fare'],
         color='red', marker='o', label='Avg Fare')
ax1.set_ylabel('Total Trips')
ax2.set_ylabel('Average Fare ($)')
ax1.set_title('Q5: Monthly Trip Volume and Average Fare', fontsize=13, fontweight='bold')
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1+lines2, labels1+labels2, loc='upper right')
plt.tight_layout()
plt.savefig('q5_monthly_trend.png', dpi=150)
plt.show()
monthly_trend.show()

In [ ]:
# ── Performance Optimisation Demo: Broadcast Join ───────────────────────────
from pyspark.sql.functions import broadcast
from pyspark.sql.types import IntegerType, StringType, StructType, StructField

# Create a small zone lookup table (simulated – replace with real CSV if available)
zone_data = [(i, f'Zone_{i}', 'Manhattan') for i in range(1, 264)]
zone_schema = StructType([
    StructField('LocationID', IntegerType()),
    StructField('Zone', StringType()),
    StructField('Borough', StringType())
])
zones = spark.createDataFrame(zone_data, schema=zone_schema)

# Broadcast join eliminates shuffle on the large DataFrame
df_with_zones = df.join(
    broadcast(zones),
    df.PULocationID == zones.LocationID,
    'left'
)

print('Physical plan (confirms BroadcastHashJoin):')
df_with_zones.explain()  # You should see BroadcastHashJoin in the plan

In [ ]:
# ── Performance Optimisation Demo: Coalesce after filtering ─────────────────
# After heavy filtering, many small partitions remain – coalesce reduces overhead
print(f'Partitions before coalesce: {df.rdd.getNumPartitions()}')
df_coalesced = df.coalesce(20)
print(f'Partitions after coalesce:  {df_coalesced.rdd.getNumPartitions()}')

---
# TASK 3 – Distributed Clustering (25%)

In [ ]:
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator
from pyspark.ml import Pipeline

# ── 3.1 Define features for trip-type clustering ────────────────────────────
feature_cols = ['fare_amount', 'trip_distance', 'trip_duration_min',
                'tip_amount', 'pickup_hour']

assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol='raw_features',
    handleInvalid='skip'
)

scaler = StandardScaler(
    inputCol='raw_features',
    outputCol='features',
    withMean=True,
    withStd=True
)

print('✅ VectorAssembler and StandardScaler configured')

In [ ]:
# ── 3.2 Elbow method – find optimal k ──────────────────────────────────────
# Note: This cell takes several minutes as it fits 7 KMeans models
wcsse_list = []
silhouette_list = []
evaluator = ClusteringEvaluator(
    predictionCol='cluster',
    featuresCol='features',
    metricName='silhouette',
    distanceMeasure='squaredEuclidean'
)

# Use a sample to speed up elbow analysis in Colab
df_sample = df.sample(fraction=0.15, seed=42).persist()
print(f'Sample size for elbow analysis: {df_sample.count():,}')

for k in range(2, 9):
    km = KMeans(featuresCol='features', predictionCol='cluster',
                k=k, seed=42, maxIter=20)
    pipeline_k = Pipeline(stages=[assembler, scaler, km])
    model_k = pipeline_k.fit(df_sample)
    preds_k = model_k.transform(df_sample)
    wcsse = model_k.stages[-1].summary.trainingCost
    sil = evaluator.evaluate(preds_k)
    wcsse_list.append(wcsse)
    silhouette_list.append(sil)
    print(f'k={k} | WCSSE={wcsse:,.0f} | Silhouette={sil:.4f}')

df_sample.unpersist()

In [ ]:
# ── 3.3 Plot elbow and silhouette curves ────────────────────────────────────
k_values = list(range(2, 9))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(k_values, wcsse_list, 'bo-')
ax1.set_title('Elbow Method – WCSSE vs k', fontweight='bold')
ax1.set_xlabel('Number of Clusters (k)')
ax1.set_ylabel('Within-Cluster SSE')

ax2.plot(k_values, silhouette_list, 'ro-')
ax2.set_title('Silhouette Score vs k', fontweight='bold')
ax2.set_xlabel('Number of Clusters (k)')
ax2.set_ylabel('Silhouette Score')

plt.tight_layout()
plt.savefig('t3_elbow_silhouette.png', dpi=150)
plt.show()
print('Choose k at the elbow point in the left chart.')

In [ ]:
# ── 3.4 Fit final KMeans model (change FINAL_K based on elbow chart) ────────
FINAL_K = 4  # ← Adjust if your elbow chart suggests a different k

km_final = KMeans(featuresCol='features', predictionCol='cluster',
                  k=FINAL_K, seed=42, maxIter=30)
pipeline_final = Pipeline(stages=[assembler, scaler, km_final])
model_final = pipeline_final.fit(df)
df_clustered = model_final.transform(df)

# Cache for use in Task 4
df_clustered.persist(StorageLevel.MEMORY_AND_DISK)
print(f'✅ KMeans fitted with k={FINAL_K}')

In [ ]:
# ── 3.5 Evaluate on full dataset ────────────────────────────────────────────
final_silhouette = evaluator.evaluate(df_clustered)
print(f'Final Silhouette Score (k={FINAL_K}): {final_silhouette:.4f}')

In [ ]:
# ── 3.6 Cluster profile analysis ────────────────────────────────────────────
cluster_stats = df_clustered.groupBy('cluster') \
    .agg(
        F.count('*').alias('n_trips'),
        F.avg('fare_amount').alias('avg_fare'),
        F.avg('trip_distance').alias('avg_distance_mi'),
        F.avg('trip_duration_min').alias('avg_duration_min'),
        F.avg('tip_amount').alias('avg_tip'),
        F.avg('pickup_hour').alias('avg_pickup_hour')
    ).orderBy('avg_fare')

cluster_stats.show()
pdf_clusters = cluster_stats.toPandas()

# Visualise cluster profiles
metrics = ['avg_fare', 'avg_distance_mi', 'avg_duration_min', 'avg_tip']
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, metric in zip(axes, metrics):
    ax.bar(pdf_clusters['cluster'].astype(str), pdf_clusters[metric], color='steelblue')
    ax.set_title(metric.replace('_',' ').title(), fontsize=11)
    ax.set_xlabel('Cluster')
plt.suptitle(f'Cluster Profiles (k={FINAL_K})', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('t3_cluster_profiles.png', dpi=150)
plt.show()

---
# TASK 4 – Distributed Supervised Learning (25%)

In [ ]:
# ── 4.1 Create pseudo-labels: synthetic demand tiers ───────────────────────
from pyspark.sql.functions import broadcast

# Count trips per hour to define demand level
hour_volume = df.groupBy('pickup_hour') \
    .agg(F.count('*').alias('hourly_volume'))

# Join back onto clustered DataFrame
df_labeled = df_clustered.join(broadcast(hour_volume), 'pickup_hour', 'left')

# Compute quantile thresholds for low / medium / high demand
q33, q66 = df_labeled.approxQuantile('hourly_volume', [0.33, 0.66], 0.01)
print(f'Demand thresholds: Low < {q33:.0f}  |  Medium < {q66:.0f}  |  High >= {q66:.0f}')

df_labeled = df_labeled.withColumn('demand_label',
    when(col('hourly_volume') <= q33, 0.0)
    .when(col('hourly_volume') <= q66, 1.0)
    .otherwise(2.0)
)

df_labeled.groupBy('demand_label').count().orderBy('demand_label').show()

In [ ]:
# ── 4.2 Feature engineering for supervised learning ─────────────────────────
supervised_features = [
    'fare_amount', 'trip_distance', 'trip_duration_min',
    'passenger_count', 'pickup_hour', 'pickup_day',
    'pickup_month', 'payment_type', 'tip_amount'
]

assembler_sl = VectorAssembler(
    inputCols=supervised_features,
    outputCol='features_sl',
    handleInvalid='skip'
)

# ── 4.3 Train / test split ──────────────────────────────────────────────────
train_df, test_df = df_labeled.randomSplit([0.8, 0.2], seed=42)
train_df.persist(StorageLevel.MEMORY_AND_DISK)
test_df.persist(StorageLevel.MEMORY_AND_DISK)

print(f'Training rows: {train_df.count():,}')
print(f'Test rows:     {test_df.count():,}')

In [ ]:
# ── 4.4 Decision Tree Classifier ────────────────────────────────────────────
from pyspark.ml.classification import DecisionTreeClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

dt = DecisionTreeClassifier(
    labelCol='demand_label',
    featuresCol='features_sl',
    maxDepth=8,
    maxBins=32,
    seed=42
)

pipeline_dt = Pipeline(stages=[assembler_sl, dt])
model_dt = pipeline_dt.fit(train_df)
predictions_dt = model_dt.transform(test_df)

print('✅ Decision Tree trained')

In [ ]:
# ── 4.5 Logistic Regression Classifier ─────────────────────────────────────
from pyspark.ml.classification import LogisticRegression

lr = LogisticRegression(
    labelCol='demand_label',
    featuresCol='features_sl',
    maxIter=50,
    regParam=0.01,
    elasticNetParam=0.0,  # L2 regularisation
    family='multinomial'
)

pipeline_lr = Pipeline(stages=[assembler_sl, lr])
model_lr = pipeline_lr.fit(train_df)
predictions_lr = model_lr.transform(test_df)

print('✅ Logistic Regression trained')

In [ ]:
# ── 4.6 Evaluate both models ────────────────────────────────────────────────
evaluator_mc = MulticlassClassificationEvaluator(
    labelCol='demand_label',
    predictionCol='prediction'
)

metrics = ['accuracy', 'f1', 'weightedPrecision', 'weightedRecall']
results = []

for metric in metrics:
    dt_score = evaluator_mc.evaluate(
        predictions_dt, {evaluator_mc.metricName: metric})
    lr_score = evaluator_mc.evaluate(
        predictions_lr, {evaluator_mc.metricName: metric})
    results.append({'Metric': metric, 'Decision Tree': dt_score, 'Logistic Regression': lr_score})
    print(f'{metric:20s}  |  DT: {dt_score:.4f}  |  LR: {lr_score:.4f}')

results_df = pd.DataFrame(results)

In [ ]:
# ── 4.7 Plot model comparison ───────────────────────────────────────────────
x = range(len(metrics))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 5))
bars1 = ax.bar([i - width/2 for i in x], results_df['Decision Tree'],
               width, label='Decision Tree', color='steelblue')
bars2 = ax.bar([i + width/2 for i in x], results_df['Logistic Regression'],
               width, label='Logistic Regression', color='coral')
ax.set_xticks(list(x))
ax.set_xticklabels(metrics)
ax.set_ylim(0, 1)
ax.set_ylabel('Score')
ax.set_title('Model Performance Comparison: DT vs Logistic Regression',
             fontsize=13, fontweight='bold')
ax.legend()
ax.bar_label(bars1, fmt='%.3f', padding=3, fontsize=9)
ax.bar_label(bars2, fmt='%.3f', padding=3, fontsize=9)
plt.tight_layout()
plt.savefig('t4_model_comparison.png', dpi=150)
plt.show()

In [ ]:
# ── 4.8 Feature importance from Decision Tree ───────────────────────────────
import numpy as np

importances = model_dt.stages[-1].featureImportances.toArray()
importance_df = pd.DataFrame({
    'Feature': supervised_features,
    'Importance': importances
}).sort_values('Importance', ascending=False)

fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(importance_df['Feature'][::-1], importance_df['Importance'][::-1],
        color='mediumseagreen')
ax.set_title('Decision Tree – Feature Importances', fontsize=13, fontweight='bold')
ax.set_xlabel('Importance Score')
plt.tight_layout()
plt.savefig('t4_feature_importance.png', dpi=150)
plt.show()

print(importance_df.to_string(index=False))

In [ ]:
# ── Clean up: unpersist all cached DataFrames ───────────────────────────────
df.unpersist()
df_clustered.unpersist()
train_df.unpersist()
test_df.unpersist()
spark.stop()
print('✅ Spark session closed. All done!')